In [2]:
"""
통계청 온라인 수집 가격 정보 API → MySQL 저장 스크립트
- 엔드포인트: http://apis.data.go.kr/1240000/bpp_openapi/getPriceInfo
- 파라미터: itemCode, startDate, endDate
- 갱신: 주 1회 (D-2일까지 조회 가능)
"""

import requests
import xml.etree.ElementTree as ET
import mysql.connector
from mysql.connector import Error
from datetime import datetime, timedelta
import time
import logging

# ─────────────────────────────────────────────
# 설정값
# ─────────────────────────────────────────────
API_KEY = "e503e06ce9147b59d04cb64c2eee1045c66ac1993f32c31806732a85a33db5c4"
BASE_URL = "http://apis.data.go.kr/1240000/bpp_openapi"

DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "database": "cooking_db",
    "user": "root",
    "password": "root",
    "charset": "utf8mb4"
}

# ─────────────────────────────────────────────
# 로깅
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("fetch_prices_kostat.log", encoding="utf-8")
    ]
)
log = logging.getLogger(__name__)


# ─────────────────────────────────────────────
# ingredients 재료명 → 통계청 품목코드 매핑
# (아까 getPriceItemList 결과 기반)
# ─────────────────────────────────────────────
INGREDIENT_TO_ITEM_CODE = {
    # 곡류
    '쌀':              'A01101',
    '밥':              'A01101',
    '현미':            'A01102',
    '찹쌀':            'A01103',
    '찹쌀가루':        'A01103',
    '보리쌀':          'A01104',
    '콩':              'A01105',
    '땅콩':            'A01106',
    '땅콩버터':        'A01106',
    '밀가루':          'A01108',
    '중력분':          'A01108',
    '강력분':          'A01108',
    '부침가루':        'A01108',
    '튀김가루':        'A01108',
    '소면':            'A01109',
    '건소면':          'A01109',
    '국수':            'A01109',
    '칼국수면':        'A01109',
    '가락국수면':      'A01109',
    '우동면':          'A01109',
    '라면':            'A01110',
    '당면':            'A01111',
    '건당면':          'A01111',
    '불린당면':        'A01111',
    '빵':              'A01116',
    '식빵':            'A01116',
    '바게트':          'A01116',
    '파스타면':        'A01118',
    '스파게티면':      'A01118',

    # 육류/가공육
    '소시지':          'A01205',
    '스팸':            'A01205',
    '햄':              'A01205',

    # 수산물
    '고등어':          'A01304',
    '오징어':          'A01305',
    '전복':            'A01309',
    '마른멸치':        'A01311',
    '멸치':            'A01311',
    '북어채':          'A01315',
    '황태채':          'A01315',

    # 유제품
    '우유':            'A01402',
    '생크림':          'A01402',
    '치즈':            'A01403',
    '모짜렐라 치즈':   'A01403',
    '슬라이스치즈':    'A01403',

    # 유지류
    '참기름':          'A01501',
    '식용유':          'A01502',
    '올리브오일':      'A01502',

    # 과일
    '사과':            'A01601',
    '배':              'A01602',
    '밤':              'A01605',
    '참외':            'A01609',
    '수박':            'A01610',
    '딸기':            'A01611',
    '키위':            'A01613',
    '블루베리':        'A01614',

    # 채소/기타농산물
    '상추':            'A01702',
    '고구마':          'A01712',
    '도라지':          'A01713',
    '풋고추':          'A01717',
    '청양고추':        'A01717',
    '홍고추':          'A01717',
    '오이고추':        'A01717',
    '호박':            'A01718',
    '애호박':          'A01718',
    '돼지호박':        'A01718',
    '토마토':          'A01720',
    '방울토마토':      'A01720',
    '대파':            'A01721',
    '파':              'A01721',
    '쪽파':            'A01721',
    '파채':            'A01721',
    '고사리':          'A01725',
    '불린고사리':      'A01725',
    '파프리카':        'A01726',
    '단무지':          'A01727',
    '김':              'A01728',
    '맛김':            'A01729',
    '미역':            'A01730',
    '불린 미역':       'A01730',
    '건 미역':         'A01730',
    '자른미역':        'A01730',

    # 과자/빙과
    '아이스크림':      'A01804',
    '스낵과자':        'A01806',
    '설탕':            'A01808',
    '황설탕':          'A01808',

    # 조미료/양념
    '고춧가루':        'A01901',
    '굵은 고춧가루':   'A01901',
    '고운 고춧가루':   'A01901',
    '생강':            'A01903',
    '소금':            'A01904',
    '꽃소금':          'A01904',
    '간장':            'A01905',
    '된장':            'A01906',
    '재래식 된장':     'A01906',
    '고추장':          'A01908',
    '카레가루':        'A01909',
    '고형카레':        'A01909',
    '식초':            'A01910',
    '양조식초':        'A01910',
    '드레싱':          'A01911',
    '혼합조미료':      'A01912',
    '다시다':          'A01912',
    '김치':            'A01915',
    '배추김치':        'A01915',
    '깍두기':          'A01915',
    '냉동식품':        'A01917',
    '즉석식품':        'A01918',
    '즉석밥':          'A01918',

    # 음료
    '커피':            'A02101',
    '생수':            'A02203',
    '탄산음료':        'A02205',
    '사이다':          'A02205',
}


# ─────────────────────────────────────────────
# 날짜 헬퍼 (D-2일까지만 조회 가능)
# ─────────────────────────────────────────────
def get_search_date() -> str:
    d = datetime.today() - timedelta(days=2)
    return d.strftime("%Y%m%d")


# ─────────────────────────────────────────────
# 가격 조회
# ─────────────────────────────────────────────
def get_prices(item_code: str, date: str) -> list:
    params = {
        "serviceKey": API_KEY,
        "itemCode":   item_code,
        "startDate":  date,
        "endDate":    date,
        "numOfRows":  "1000",
        "pageNo":     "1"
    }
    try:
        resp = requests.get(f"{BASE_URL}/getPriceInfo", params=params, timeout=15)
        resp.raise_for_status()
        root = ET.fromstring(resp.content)

        # 에러 코드 확인
        result_code = root.findtext(".//resultCode") or ""
        if result_code != "00":
            result_msg = root.findtext(".//resultMsg") or ""
            log.warning(f"API 오류 [{item_code}]: {result_code} - {result_msg}")
            return []

        prices = []
        for item in root.iter("item"):
            try:
                sp = item.findtext("sp") or "0"   # 판매가격
                dp = item.findtext("dp") or "0"   # 할인가격
                bp = item.findtext("bp") or "0"   # 혜택가격
                pn = item.findtext("pn") or ""    # 상품명
                sd = item.findtext("sd") or date  # 가격일자

                sp = float(sp.replace(",", ""))
                dp = float(dp.replace(",", ""))

                if sp > 0:
                    prices.append({
                        "item_code":    item_code,
                        "product_name": pn,
                        "sell_price":   sp,
                        "disc_price":   dp,
                        "price_date":   sd
                    })
            except (ValueError, TypeError):
                continue
        return prices

    except requests.RequestException as e:
        log.error(f"HTTP 오류 [{item_code}]: {e}")
        return []
    except ET.ParseError as e:
        log.error(f"XML 파싱 오류 [{item_code}]: {e}")
        return []


# ─────────────────────────────────────────────
# DB 테이블 생성
# ─────────────────────────────────────────────
def ensure_tables(cursor):
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS ingredient_prices (
            id              INT AUTO_INCREMENT PRIMARY KEY,
            ingredient_id   INT NOT NULL,
            item_code       VARCHAR(20),
            product_name    VARCHAR(300),
            sell_price      DECIMAL(10,2),
            disc_price      DECIMAL(10,2),
            price_date      DATE,
            created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            INDEX idx_ingredient_id (ingredient_id),
            INDEX idx_price_date (price_date)
        ) CHARACTER SET utf8mb4
    """)


# ─────────────────────────────────────────────
# 메인
# ─────────────────────────────────────────────
def main():
    search_date = get_search_date()
    log.info(f"=== 가격 수집 시작 | 조회일: {search_date} ===")

    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor(dictionary=True)
        log.info("DB 연결 성공")
    except Error as e:
        log.error(f"DB 연결 실패: {e}")
        return

    try:
        ensure_tables(cursor)
        conn.commit()

        # 기존 데이터 삭제 (오늘 날짜 기준 재수집)
        cursor.execute("DELETE FROM ingredient_prices WHERE price_date = %s", (search_date,))
        conn.commit()

        cursor.execute("SELECT id, name FROM ingredients ORDER BY name")
        ingredients = cursor.fetchall()
        log.info(f"재료 {len(ingredients)}개 로드")

        total_saved = 0
        matched_count = 0
        unmatched = []

        for ing in ingredients:
            ing_id   = ing["id"]
            ing_name = ing["name"]

            item_code = INGREDIENT_TO_ITEM_CODE.get(ing_name)
            if not item_code:
                unmatched.append(ing_name)
                continue

            matched_count += 1
            prices = get_prices(item_code, search_date)
            time.sleep(0.1)

            if not prices:
                log.warning(f"[{ing_name}] ({item_code}) → 가격 데이터 없음")
                continue

            # 중간값 가격으로 avg_price 계산
            sell_prices = [p["sell_price"] for p in prices]
            sell_prices.sort()
            mid = len(sell_prices) // 2
            avg = sell_prices[mid] if len(sell_prices) % 2 != 0 else \
                  (sell_prices[mid-1] + sell_prices[mid]) / 2

            for p in prices:
                cursor.execute("""
                    INSERT INTO ingredient_prices
                        (ingredient_id, item_code, product_name, sell_price, disc_price, price_date)
                    VALUES (%s, %s, %s, %s, %s, %s)
                """, (ing_id, p["item_code"], p["product_name"],
                      p["sell_price"], p["disc_price"], p["price_date"]))
                total_saved += 1

            cursor.execute(
                "UPDATE ingredients SET avg_price = %s WHERE id = %s",
                (round(avg, 2), ing_id)
            )
            log.info(f"[{ing_name}] ({item_code}) → {len(prices)}건 | 중간값 {avg:,.0f}원")

        conn.commit()

        log.info(f"=== 완료 ===")
        log.info(f"매핑된 재료: {matched_count}개 / 미매핑: {len(unmatched)}개")
        log.info(f"총 가격 데이터: {total_saved}건 저장")
        if unmatched:
            log.info(f"미매핑 재료 (상위 30개): {unmatched[:30]}")

    except Error as e:
        log.error(f"DB 오류: {e}")
        conn.rollback()
    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    main()

2026-03-04 14:19:41,645 [INFO] === 가격 수집 시작 | 조회일: 20260302 ===


2026-03-04 14:19:41,664 [INFO] DB 연결 성공
2026-03-04 14:19:41,843 [INFO] 재료 707개 로드
2026-03-04 14:19:43,697 [WARNING] API 오류 [A01109]: 21 - THERE IS NO MSG IN YOUR REQUEST.
2026-03-04 14:19:43,801 [WARNING] [가락국수면] (A01109) → 가격 데이터 없음
2026-03-04 14:19:43,993 [WARNING] API 오류 [A01905]: 21 - THERE IS NO MSG IN YOUR REQUEST.
2026-03-04 14:19:44,096 [WARNING] [간장] (A01905) → 가격 데이터 없음
2026-03-04 14:19:46,464 [WARNING] API 오류 [A01108]: 21 - THERE IS NO MSG IN YOUR REQUEST.
2026-03-04 14:19:46,569 [WARNING] [강력분] (A01108) → 가격 데이터 없음
2026-03-04 14:19:46,662 [WARNING] API 오류 [A01730]: 21 - THERE IS NO MSG IN YOUR REQUEST.
2026-03-04 14:19:46,768 [WARNING] [건 미역] (A01730) → 가격 데이터 없음
2026-03-04 14:19:48,187 [WARNING] API 오류 [A01712]: 21 - THERE IS NO MSG IN YOUR REQUEST.
2026-03-04 14:19:48,294 [WARNING] [고구마] (A01712) → 가격 데이터 없음
2026-03-04 14:19:48,372 [WARNING] API 오류 [A01901]: 21 - THERE IS NO MSG IN YOUR REQUEST.
2026-03-04 14:19:48,477 [WARNING] [고운 고춧가루] (A01901) → 가격 데이터 없음
2026-03-04 1